# Тестирование "деревянных" моделей и бустингов.

In [2]:
import polars as pl
import math
from main_funcs import global_temporal_split, ndcg_at_k, calculate_metrics

In [3]:
from catboost import CatBoostRanker, Pool
import optuna

/home/alex/VS_python_projects/DreamTeamRecSys/.venv313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
events_path = "../../data/dataset/small/marketplace/events"
items_path = "../../data/dataset/small/marketplace/items.pq"

In [5]:
events_df = pl.scan_parquet(events_path)
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String)])

In [6]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os
duration[μs],u64,str,str,str,str
1082d 89901µs,59328774,"""nfmcg_14777696""","""u2i""","""view""","""android"""
1082d 163483µs,66295302,"""nfmcg_26098539""","""catalog""","""view""","""android"""
1082d 864625µs,37542104,"""nfmcg_10840247""","""u2i""","""view""","""android"""
1082d 889192µs,35193548,"""nfmcg_8040572""","""catalog""","""view""","""android"""
1082d 936008µs,27256137,"""nfmcg_6770239""","""catalog""","""view""","""android"""


In [7]:
actions_count = events_df.group_by("action_type").agg(pl.len()).collect(engine="streaming")
actions_count.head()

action_type,len
str,u32
"""view""",126147783
"""clickout""",485448
"""like""",41792
"""click""",3772094


Классы явно не сбалансированны, просмотров гораздо больше. Чтоб сформировать таргет, нужно превратить частоту действия в числовой таргет так, чтобы редкие действия получали больший вес, а частые - меньший. Общее число наблюдений мы делим на число наблюдений этого конкретного действия.

Беру логарифмированный таргет, так как он показал наилучшие результаты в бейзлайне.

In [8]:
actions_count = actions_count.with_columns(
    (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target") 
).drop("len")
actions_count

action_type,log_target
str,f64
"""view""",0.710044
"""clickout""",5.597366
"""like""",8.046339
"""click""",3.571844


In [9]:
events_df = events_df.join(actions_count.lazy(), on="action_type").with_columns([
    (pl.col("action_type") == "view").cast(pl.Int8).alias("target_view"),
    (pl.col("action_type") == "clickout").cast(pl.Int8).alias("target_clickout"),
    (pl.col("action_type") == "like").cast(pl.Int8).alias("target_like"),
    (pl.col("action_type") == "click").cast(pl.Int8).alias("target_click"),
])
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,log_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,f64,i8,i8,i8,i8
1082d 14h 4m 26s 1ms,76807164,"""nfmcg_9979701""","""catalog""","""view""","""android""",0.710044,1,0,0,0
1082d 14h 4m 26s 134086µs,43753743,"""nfmcg_8421453""","""catalog""","""view""","""android""",0.710044,1,0,0,0
1082d 14h 4m 26s 256221µs,50022940,"""nfmcg_12666278""","""search""","""view""","""android""",0.710044,1,0,0,0
1082d 14h 4m 26s 369014µs,83768122,"""nfmcg_2104169""","""search""","""view""","""ios""",0.710044,1,0,0,0
1082d 14h 4m 26s 441077µs,7859519,"""nfmcg_2138782""","""u2i""","""view""","""android""",0.710044,1,0,0,0


In [10]:
null_info = events_df.null_count().collect()
print(null_info)

shape: (1, 11)
┌───────────┬─────────┬─────────┬───────────┬───┬────────────┬────────────┬────────────┬───────────┐
│ timestamp ┆ user_id ┆ item_id ┆ subdomain ┆ … ┆ target_vie ┆ target_cli ┆ target_lik ┆ target_cl │
│ ---       ┆ ---     ┆ ---     ┆ ---       ┆   ┆ w          ┆ ckout      ┆ e          ┆ ick       │
│ u32       ┆ u32     ┆ u32     ┆ u32       ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---       │
│           ┆         ┆         ┆           ┆   ┆ u32        ┆ u32        ┆ u32        ┆ u32       │
╞═══════════╪═════════╪═════════╪═══════════╪═══╪════════════╪════════════╪════════════╪═══════════╡
│ 0         ┆ 0       ┆ 0       ┆ 41792     ┆ … ┆ 0          ┆ 0          ┆ 0          ┆ 0         │
└───────────┴─────────┴─────────┴───────────┴───┴────────────┴────────────┴────────────┴───────────┘


Посмотрю по таблице items, какие есть пропуски.

In [11]:
ivents_lf = pl.scan_parquet(items_path)
ivents_lf.head().collect(engine="streaming")

item_id,brand_id,category,subcategory,price,embedding
str,u64,str,str,f64,"array[f32, 300]"
"""nfmcg_10000001""",137356,null,null,6.06364,"[-0.070853, 0.03246, … 0.082178]"
"""nfmcg_10000012""",53389,null,null,0.960436,"[-0.091099, 0.043168, … 0.060287]"
"""nfmcg_1000002""",34153,"""Fashion Accessories, Tech Add-…","""Hats, Scarves, and Shawls""",-1.793113,"[-0.056533, 0.082878, … 0.037475]"
"""nfmcg_10000034""",39543,null,null,3.416755,"[-0.112588, 0.043333, … -0.001615]"
"""nfmcg_10000039""",82320,"""Fashion Accessories, Tech Add-…","""Jewelry and Costume Jewelry""",4.293433,"[-0.157201, 0.026234, … 0.013904]"


In [12]:
null_info = ivents_lf.null_count().collect()
print("Количество строк:", ivents_lf.select(pl.len()).collect().item())
print("Количество колонок:", len(ivents_lf.columns))
print(null_info)

Количество строк: 2325409
Количество колонок: 6
shape: (1, 6)
┌─────────┬──────────┬──────────┬─────────────┬───────┬───────────┐
│ item_id ┆ brand_id ┆ category ┆ subcategory ┆ price ┆ embedding │
│ ---     ┆ ---      ┆ ---      ┆ ---         ┆ ---   ┆ ---       │
│ u32     ┆ u32      ┆ u32      ┆ u32         ┆ u32   ┆ u32       │
╞═════════╪══════════╪══════════╪═════════════╪═══════╪═══════════╡
│ 0       ┆ 0        ┆ 966395   ┆ 1233023     ┆ 2882  ┆ 73        │
└─────────┴──────────┴──────────┴─────────────┴───────┴───────────┘


/tmp/ipykernel_433252/2265000357.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок:", len(ivents_lf.columns))


Подготовлю данные для дальнейшей работы с моделями. Категориальные признаки оставляю, так как бустинги умеют с ними работать. Удаляю товары с пропусками в цене, так как их минимальное количество от общего числа. Эмбеддинги пока не беру, пропуски в категориях и подкатегориях заменяю на новую категорию "unknown". Строки в таблице действий с пропусками в поддомене также просто удаляю, так как их не много. Далее форматирую временную отметку в таблоице событий и собираю все в одну таблицу events_prepared, с которой и продолжу работу.

In [13]:
ACCEPTABLE_ACTIONS = ["view", "click", "clickout", "like"]

# Очистка items
items_lf = (
    ivents_lf
    .drop_nulls(subset=["price"])
    .with_columns([
        pl.col("category").fill_null("unknown").alias("category"),
        pl.col("subcategory").fill_null("unknown").alias("subcategory"),
        pl.col("brand_id").fill_null(-1).alias("brand_id"),
    ])
    .select([
        "item_id",
        "brand_id",
        "category",
        "subcategory",
        "price",
        # embedding пока не использую
    ])
)

# Подготовка events
MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000

events_prepared = (
    pl.scan_parquet(events_path)
    .filter(pl.col("action_type").is_in(ACCEPTABLE_ACTIONS))
    .join(actions_count.lazy(), on="action_type", how="inner")
    .join(items_lf, on="item_id", how="inner")
    .with_columns([
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).alias("day"),
    ])
    .filter(pl.col("subdomain").is_not_null())
    .select([
        "timestamp",
        "day",
        "user_id",
        "item_id",
        "subdomain",
        "action_type",
        "os",
        "log_target",
        "brand_id",
        "category",
        "subcategory",
        "price",
    ])
)

In [14]:
events_prepared.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('day', Int64),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('log_target', Float64),
        ('brand_id', Int64),
        ('category', String),
        ('subcategory', String),
        ('price', Float64)])

In [15]:
events_prepared.head().collect(engine="streaming")

timestamp,day,user_id,item_id,subdomain,action_type,os,log_target,brand_id,category,subcategory,price
duration[μs],i64,u64,str,str,str,str,f64,i64,str,str,f64
1082d 15h 50m 28s 471134µs,1082,82107839,"""nfmcg_3444394""","""search""","""click""","""ios""",3.571844,117885,"""Electronic Devices and Gadgets""","""Portable Electronics""",4.275868
1082d 15h 51m 51s 850727µs,1082,3756069,"""nfmcg_2992071""","""u2i""","""click""","""android""",3.571844,173015,"""Cosmetics, Personal Care, and …","""Perfumes and Aromatic Products""",1.723022
1082d 15h 53m 49s 875402µs,1082,45560762,"""nfmcg_6788092""","""u2i""","""click""","""android""",3.571844,117885,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",5.783138
1082d 15h 56m 9s 457265µs,1082,71394359,"""nfmcg_6788092""","""u2i""","""click""","""android""",3.571844,117885,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",5.783138
1082d 15h 59m 49s 965882µs,1082,73983736,"""nfmcg_26281589""","""u2i""","""click""","""android""",3.571844,211031,"""Cosmetics, Personal Care, and …","""Perfumes and Aromatic Products""",0.004676


Беру данные только за 5% последних дней - максимум, который позволяет моя оперативная память (32гб).

In [16]:
#границы дней
raw_events = pl.scan_parquet(events_path)
MICROS_IN_DAY = 24 * 60 * 60 * 1_000_000

min_day, max_day = (
    raw_events
    .select(
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).min().alias("min_day"),
        (pl.col("timestamp").cast(pl.Int64) // MICROS_IN_DAY).max().alias("max_day"),
    )
    .collect()
    .row(0)
)

days_all = (max_day - min_day) + 1
n_days_subset = int(days_all * 0.05)
if n_days_subset < 1:
    n_days_subset = 1

subset_min_day = max_day - n_days_subset + 1

events_subset = events_prepared.filter(pl.col("day") >= subset_min_day)

In [17]:
train, test = global_temporal_split(events_subset, 0.2)
print("Количество строк train:", train.select(pl.len()).collect().item())
print("Количество колонок train:", len(train.columns))
print("Количество строк test:", test.select(pl.len()).collect().item())
print("Количество колонок test:", len(test.columns))

Количество строк train: 4044137
Количество колонок train: 12


/tmp/ipykernel_433252/3991756773.py:3: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок train:", len(train.columns))


Количество строк test: 1072961
Количество колонок test: 12


/tmp/ipykernel_433252/3991756773.py:5: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print("Количество колонок test:", len(test.columns))


In [18]:
# агрегируем события до уровня user-item в train и test
train_ui = (
    train
    .group_by(["user_id", "item_id"])
    .agg([
        pl.col("log_target").max().alias("target"),
        pl.col("day").max().alias("last_day"),       
        pl.col("price").mean().alias("price_mean"),
        pl.col("category").first().alias("category"),
        pl.col("subcategory").first().alias("subcategory"),
        pl.col("brand_id").first().alias("brand_id"),
        pl.col("os").first().alias("os"),
        pl.col("subdomain").first().alias("subdomain"),
        pl.col("action_type").first().alias("last_action_type"),
    ])
)

test_ui = (
    test
    .group_by(["user_id", "item_id"])
    .agg([
        pl.col("log_target").max().alias("target"),
        pl.col("day").max().alias("last_day"),
        pl.col("price").mean().alias("price_mean"),
        pl.col("category").first().alias("category"),
        pl.col("subcategory").first().alias("subcategory"),
        pl.col("brand_id").first().alias("brand_id"),
        pl.col("os").first().alias("os"),
        pl.col("subdomain").first().alias("subdomain"),
        pl.col("action_type").first().alias("last_action_type"),
    ])
)

In [19]:
train_ui.head().collect(engine="streaming")

user_id,item_id,target,last_day,price_mean,category,subcategory,brand_id,os,subdomain,last_action_type
u64,str,f64,i64,f64,str,str,i64,str,str,str
81646445,"""nfmcg_15616899""",0.710044,1300,5.435182,"""Electronic Devices and Gadgets""","""Mobile Devices and Electronic …",28667,"""other""","""catalog""","""view"""
79067791,"""nfmcg_22146622""",0.710044,1300,4.923074,"""Electronic Devices and Gadgets""","""Gaming Electronic Devices""",137356,"""android""","""search""","""view"""
18819130,"""nfmcg_8018741""",0.710044,1300,-0.058052,"""unknown""","""unknown""",241818,"""android""","""i2i""","""view"""
37575338,"""nfmcg_8961427""",0.710044,1300,2.619599,"""Home/Office Furniture and Inte…","""Kitchen Sets and Interior Elem…",198171,"""android""","""u2i""","""view"""
27114721,"""nfmcg_20992168""",0.710044,1300,2.31735,"""Electronic Devices and Gadgets""","""Portable Audio Electronics""",137356,"""other""","""catalog""","""view"""


In [20]:
test_ui.head().collect(engine="streaming")

user_id,item_id,target,last_day,price_mean,category,subcategory,brand_id,os,subdomain,last_action_type
u64,str,f64,i64,f64,str,str,i64,str,str,str
63924551,"""nfmcg_26613208""",3.571844,1307,-1.69912,"""Outerwear, Casual Apparel, and…","""Girls' Children's Clothing""",189615,"""other""","""catalog""","""click"""
62663545,"""nfmcg_3833""",0.710044,1307,-0.786328,"""unknown""","""unknown""",211031,"""other""","""catalog""","""view"""
28771653,"""nfmcg_25927310""",0.710044,1307,3.429408,"""unknown""","""unknown""",232696,"""other""","""i2i""","""view"""
56194471,"""nfmcg_11223100""",0.710044,1307,-0.910935,"""unknown""","""unknown""",141256,"""android""","""other""","""view"""
41995600,"""nfmcg_15913816""",0.710044,1307,1.574822,"""unknown""","""unknown""",141256,"""other""","""u2i""","""view"""


Заведу еще валидационную выборку.

In [21]:
min_day, max_day = (
    train_ui.select(
        pl.col("last_day").min().alias("min_day"),
        pl.col("last_day").max().alias("max_day"),
    ).collect().row(0)
)

days_all = (max_day - min_day) + 1
val_days = max(int(days_all * 0.1), 1)
val_min_day = max_day - val_days + 1

train_train_lf = train_ui.filter(pl.col("last_day") < val_min_day)
valid_lf = train_ui.filter(pl.col("last_day") >= val_min_day)

Добавлю новые признаки - агрегаты по пользователям, товарам и истории взаимодействия user-item. Эти признаки позже передам для обучения модели.

In [22]:
# user/popularity агрегаты на train_train
user_aggs = (
    train_train_lf
    .group_by("user_id")
    .agg([
        pl.len().alias("user_hist_len"),
        pl.col("target").mean().alias("user_mean_target"),
    ])
)

item_aggs = (
    train_train_lf
    .group_by("item_id")
    .agg([
        pl.len().alias("item_popularity"),
        pl.col("price_mean").mean().alias("price_mean_item"),
    ])
)

In [23]:
print(user_aggs.head().collect(engine="streaming"))
print("________________________")
print(item_aggs.head().collect(engine="streaming"))

shape: (5, 3)
┌──────────┬───────────────┬──────────────────┐
│ user_id  ┆ user_hist_len ┆ user_mean_target │
│ ---      ┆ ---           ┆ ---              │
│ u64      ┆ u32           ┆ f64              │
╞══════════╪═══════════════╪══════════════════╡
│ 68766818 ┆ 3             ┆ 0.710044         │
│ 74029335 ┆ 74            ┆ 0.78739          │
│ 83493688 ┆ 15            ┆ 0.710044         │
│ 50492804 ┆ 116           ┆ 0.818979         │
│ 46082024 ┆ 5             ┆ 0.710044         │
└──────────┴───────────────┴──────────────────┘
________________________
shape: (5, 3)
┌────────────────┬─────────────────┬─────────────────┐
│ item_id        ┆ item_popularity ┆ price_mean_item │
│ ---            ┆ ---             ┆ ---             │
│ str            ┆ u32             ┆ f64             │
╞════════════════╪═════════════════╪═════════════════╡
│ nfmcg_25292855 ┆ 1               ┆ 1.352331        │
│ nfmcg_15238273 ┆ 15              ┆ 3.407273        │
│ nfmcg_3541211  ┆ 2              

In [24]:
def prep_rank_lf(base_lf: pl.LazyFrame) -> pl.LazyFrame:
    return (
        base_lf
        .join(user_aggs, on="user_id", how="left")
        .join(item_aggs, on="item_id", how="left")
    )

train_rank_lf = prep_rank_lf(train_train_lf)
valid_rank_lf = prep_rank_lf(valid_lf)

train_df = train_rank_lf.collect().to_pandas(use_pyarrow_extension_array=False)
valid_df = valid_rank_lf.collect().to_pandas(use_pyarrow_extension_array=False)

In [25]:
feature_cols = [
    "item_id",
    "price_mean",
    "price_mean_item",
    "category",
    "subcategory",
    "brand_id",
    "user_hist_len",
    "user_mean_target",
    "item_popularity",
    "os",
    "subdomain",
    "last_day",
]

cat_features = ["item_id", "category", "subcategory", "brand_id", "os", "subdomain"]

In [26]:
train_df = train_df.sort_values(["user_id", "item_id"])
valid_df = valid_df.sort_values(["user_id", "item_id"])

Обучаю первую модель- вариант CatBoost для задач ранжирования CatBoostRanker с дефолтной функцией потерь для ранжирования YetiRank, беру ее так как она хорошо работает с категориальными признаками и не требует тонких настроек гиперпараметров, можно сразу увидеть результат для сравнения с нашим бейзлайном - SVD. 

Беру только 5000 самых популярных товаров для кандидатогенерации, тк не хватает оперативной памяти.

In [ ]:
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df["target"],
    group_id=train_df["user_id"],
    cat_features=cat_features,
)

valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df["target"],
    group_id=valid_df["user_id"],
    cat_features=cat_features,
)

model = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=15",
    iterations=300,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=50,
    early_stopping_rounds=50,
    task_type="CPU",
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

In [28]:
TOP_M_ITEMS = 5000
TOP_K = 15

top_items_lf = (
    train_ui
    .group_by("item_id")
    .agg(pl.len().alias("cnt"))
    .sort("cnt", descending=True)
    .limit(TOP_M_ITEMS)
    .select("item_id")
)

user_test_truth = (
    test_ui
    .group_by("user_id")
    .agg([
        pl.col("item_id").alias("true_items"),
        pl.col("target").alias("relevancy"),
    ])
)

Реализую функцию вычисления по батчам с возможностью выбрать размер батча, число юзеров и top k товаров.

In [29]:
def get_topk_from_toppop(
    model,
    feature_cols,
    cat_features,
    user_test_truth: pl.LazyFrame,
    top_items_lf: pl.LazyFrame,
    batch_size: int = 200,
    n_test_users: int | None = None,
    top_k: int = 15,
) -> pl.DataFrame:

    base_user_ids = user_test_truth.select("user_id").unique()
    if n_test_users is not None:
        base_user_ids = base_user_ids.limit(n_test_users)

    user_ids = base_user_ids.collect()["user_id"].to_list()
    n_users = len(user_ids)
    n_batches = math.ceil(n_users / batch_size)

    all_rows = []

    for b in range(n_batches):
        s = b * batch_size
        e = min((b + 1) * batch_size, n_users)
        batch_users = user_ids[s:e]
        print(f"Batch {b+1}/{n_batches}: users {s}..{e-1}")

        truth_small = user_test_truth.filter(pl.col("user_id").is_in(batch_users))
        users_lf = truth_small.select("user_id").unique()

        # кандидаты = user × top-5000
        cand_lf = users_lf.join(top_items_lf, how="cross")

        # навешиваем те же фичи, что в train/valid
        cand_with_features = (
            cand_lf
            .join(
                train_ui.select(
                    "user_id", "item_id", "price_mean", "category",
                    "subcategory", "brand_id", "os", "subdomain", "last_day",
                ),
                on=["user_id", "item_id"],
                how="left",
            )
            .join(
                user_aggs,
                on="user_id",
                how="left",
            )
            .join(
                item_aggs,
                on="item_id",
                how="left",
            )
        )

        cols = cand_with_features.collect_schema().names()
        base_cols = ["user_id", "item_id"]
        feature_cols_present = [
            c for c in feature_cols
            if c in cols and c not in base_cols
        ]

        cand_lf_sel = cand_with_features.select(
            [pl.col(c) for c in base_cols + feature_cols_present]
        )

        cand_df = cand_lf_sel.collect().to_pandas(use_pyarrow_extension_array=False)

        for col in cat_features:
            cand_df[col] = cand_df[col].astype("string").fillna("__nan__")

        cand_df["user_hist_len"] = cand_df["user_hist_len"].fillna(0).astype("int32")
        cand_df["item_popularity"] = cand_df["item_popularity"].fillna(0).astype("int32")
        cand_df["user_mean_target"] = cand_df["user_mean_target"].fillna(0.0)
        cand_df["price_mean_item"] = cand_df["price_mean_item"].fillna(
            cand_df["price_mean"]
        )
        cand_df["last_day"] = cand_df["last_day"].fillna(min_day).astype("int32")

        cand_df = cand_df.sort_values(["user_id", "item_id"]).reset_index(drop=True)

        pool = Pool(
            data=cand_df[feature_cols],
            group_id=cand_df["user_id"],
            cat_features=cat_features,
        )

        cand_df["pred_score"] = model.predict(pool)

        batch_pl = pl.from_pandas(cand_df)

        batch_topk = (
            batch_pl
            .sort(["user_id", "pred_score"], descending=[False, True])
            .group_by("user_id")
            .agg([
                pl.col("item_id").head(top_k).alias("predicted_items"),
                pl.col("pred_score").head(top_k).alias("predicted_scores"),
            ])
        )

        truth_df = truth_small.collect()
        batch_eval = truth_df.join(batch_topk, on="user_id", how="inner")

        all_rows.append(batch_eval)

    return pl.concat(all_rows)

In [ ]:
topk_df = get_topk_from_toppop(
    model=model,
    feature_cols=feature_cols,
    cat_features=cat_features,
    user_test_truth=user_test_truth,
    top_items_lf=top_items_lf,
    batch_size=200,
    n_test_users=1000,
    top_k=TOP_K,
)

Batch 1/5: users 0..199
Batch 2/5: users 200..399
Batch 3/5: users 400..599
Batch 4/5: users 600..799
Batch 5/5: users 800..999


In [ ]:
ndcg_df = ndcg_at_k(
    user_based_df=topk_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=TOP_K,
)

mean_ndcg = ndcg_df["ndcg"].mean()

precision_k, recall_k = calculate_metrics(
    df=topk_df.select(["true_items", "predicted_items"]),
    k=TOP_K,
)

print(f"Test mean NDCG@{TOP_K}:", mean_ndcg)
print(f"Test Precision@{TOP_K}:", precision_k)
print(f"Test Recall@{TOP_K}:", recall_k)

Test mean NDCG@15: 0.17011341418374915
Test Precision@15: 0.0562
Test Recall@15: 0.16860786787652574


Потратил больше 6 часов, тк вычисления постоянно прерывались из-за кончающейся оперативной памяти (32гб на локальной машине), поэтому приходилось много раз подбирать количество батчей, юзеров, дней для выборки, чтоб получить хоть какие-то результаты. Из-за этого пока не хватило времени более детально подобрать гиперпараметры и протестировать другие модели бустинга. В целом качество слабое, но выше бейзлайна, что, я считаю, удовлетворительно в рамках небольшого тестирования. 

Далее подберу оптимальные гиперпараметры с помощью Optuna. Буду подбирать самые важные гиперпараметры: learning rate, глубину деревьев, коэффициент L2 регуляризации.

In [30]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "loss_function": "YetiRank",
        "eval_metric": f"NDCG:top={TOP_K}",
        "iterations": 300,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "random_seed": 42,
        "verbose": 0,
        "task_type": "CPU",
        "early_stopping_rounds": 30,
    }

    model = CatBoostRanker(**params)
    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        verbose=False,
    )

    valid_scored = valid_df.copy()
    valid_scored["pred"] = model.predict(valid_df[feature_cols])
    valid_scored = valid_scored.sort_values(["user_id", "pred"], ascending=[True, False])

    pred_df = (
        pl.from_pandas(valid_scored[["user_id", "item_id", "pred"]])
        .group_by("user_id")
        .agg(
            pl.col("item_id").head(TOP_K).alias("predicted_items"),
            pl.col("pred").head(TOP_K).alias("predicted_scores"),
        )
    )

    truth_df = (
        pl.from_pandas(valid_df[["user_id", "item_id", "target"]])
        .group_by("user_id")
        .agg(
            pl.col("item_id").alias("true_items"),
            pl.col("target").alias("relevancy"),
        )
    )

    eval_df = truth_df.join(pred_df, on="user_id", how="inner")

    ndcg_df = ndcg_at_k(
        user_based_df=eval_df,
        relevancy_col="relevancy",
        true_items_col="true_items",
        predicted_items_col="predicted_items",
        predicted_score_col="predicted_scores",
        top_k=TOP_K,
    )

    return float(ndcg_df["ndcg"].mean())

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=25,
    timeout=600,
)

In [ ]:
print("Best value:", study.best_value)
print("Best params:", study.best_params)
print("Best trial:", study.best_trial.number)

Best value: 0.9884941446299346
Best params: {'learning_rate': 0.023688639503640783, 'depth': 10, 'l2_leaf_reg': 7.587945476302646}
Best trial: 0


Теперь снова обучу модель уже с лучшими подобранными гиперпараметрами.

In [32]:
model_best = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=15",
    iterations=300,
    learning_rate=0.023,
    depth=10,
    l2_leaf_reg=7.6,
    random_seed=42,
    verbose=50,
    early_stopping_rounds=50,
    task_type="CPU",
)

model_best.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        early_stopping_rounds=50,
        verbose=50,
    )

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.9772831	best: 0.9772831 (0)	total: 1.25s	remaining: 6m 14s
50:	test: 0.9803223	best: 0.9803223 (50)	total: 1m 1s	remaining: 4m 59s
100:	test: 0.9806107	best: 0.9806107 (100)	total: 1m 59s	remaining: 3m 55s
150:	test: 0.9809678	best: 0.9809678 (150)	total: 2m 55s	remaining: 2m 52s
200:	test: 0.9811043	best: 0.9811092 (199)	total: 3m 50s	remaining: 1m 53s
250:	test: 0.9811843	best: 0.9811940 (246)	total: 4m 48s	remaining: 56.3s
299:	test: 0.9812362	best: 0.9812510 (295)	total: 5m 45s	remaining: 0us

bestTest = 0.981251032
bestIteration = 295

Shrink model to first 296 iterations.


CatBoostRanker(depth=10, early_stopping_rounds=50, eval_metric='NDCG:top=15', iterations=300, l2_leaf_reg=7.6, learning_rate=0.023, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=50)

Снова дважды кончалась память, поэтому решил изменить размер батчей. Считаю тест только на 1000 пользователях.

In [33]:
best_topk_df = get_topk_from_toppop(
    model=model_best,
    feature_cols=feature_cols,
    cat_features=cat_features,
    user_test_truth=user_test_truth,
    top_items_lf=top_items_lf,
    batch_size=100,
    n_test_users=1000,
    top_k=TOP_K,
)

Batch 1/10: users 0..99
Batch 2/10: users 100..199
Batch 3/10: users 200..299
Batch 4/10: users 300..399
Batch 5/10: users 400..499
Batch 6/10: users 500..599
Batch 7/10: users 600..699
Batch 8/10: users 700..799
Batch 9/10: users 800..899
Batch 10/10: users 900..999


In [34]:
ndcg_df_best = ndcg_at_k(
    user_based_df=best_topk_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=TOP_K,
)

mean_best_ndcg = ndcg_df_best["ndcg"].mean()

best_precision_k, best_recall_k = calculate_metrics(
    df=best_topk_df.select(["true_items", "predicted_items"]),
    k=TOP_K,
)

print(f"Best mean NDCG@{TOP_K}:", mean_best_ndcg)
print(f"Best Precision@{TOP_K}:", best_precision_k)
print(f"Best Recall@{TOP_K}:", best_recall_k)

Best mean NDCG@15: 0.3377691164036194
Best Precision@15: 0.11359999999999999
Best Recall@15: 0.35636379107844635


Итоговый результат после преминения Optuna на тесте из 5000 популярных товаров по 1000 пользователям оказался значительно лучше бейзлайна. Подбор гиперпараметров сработал корректно, однако метрики явно завышены из-за тестирования не неплоной выборке и сравнить их с бейзлайном SVD и другими моделями не получится.

# Итоговые лучшие метрики:
## NDCG@15: 0.33

## Precision@15: 0.11

## Recall@15: 0.36